# Team aEC/T Leaderboard

Rank UFA teams by possession aEC per throw. The main rate is `team_aec_per_throw = sum(total_aec) / sum(throw_count)`, which treats every throw as one unit in the team rate.

In [ ]:
import base64
import importlib
import mimetypes
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.leaderboards

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.leaderboards)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    build_possessions,
    build_scoring_possessions,
    fetch_shownspace_season_throws_cached,
    filter_analysis_possessions,
    summarize_team_aec_consistency,
    summarize_team_aec_per_throw,
    summarize_team_top_possessions,
)

In [ ]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'
MIN_POSSESSIONS = 20
TOP_TEAMS = 22
TOP_POSSESSION_COUNT = 5
TOP_POSSESSION_MIN_THROWS = 5
CONSISTENCY_TOP_N = 10
CONSISTENCY_MIN_THROWS = 5

## Load League Possessions

In [ ]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_all_possessions, league_all_paths = build_possessions(league_throws)
league_possessions, league_paths = build_scoring_possessions(league_throws)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League all possessions found: {len(league_all_possessions):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')

In [ ]:
LEADERBOARD_COLUMNS = [
    'rank', 'team_id', 'team_aec_per_throw', 'avg_possession_aec_per_throw',
    'possessions', 'games', 'throws', 'total_aec', 'avg_throw_count',
    'avg_field_progress', 'huck_rate', 'resets_per_possession', 'o_line_share',
]

FORMATTERS = {
    'team_aec_per_throw': '{:.3f}',
    'avg_possession_aec_per_throw': '{:.3f}',
    'median_possession_aec_per_throw': '{:.3f}',
    'total_aec': '{:.1f}',
    'avg_throw_count': '{:.1f}',
    'avg_field_progress': '{:.1f}',
    'huck_rate': '{:.1%}',
    'resets_per_possession': '{:.1f}',
    'o_line_share': '{:.1%}',
    'top_mean_metric': '{:.3f}',
    'top_median_metric': '{:.3f}',
    'top_floor_metric': '{:.3f}',
    'top_ceiling_metric': '{:.3f}',
    'top_total_aec': '{:.1f}',
    'top_avg_throw_count': '{:.1f}',
    'median_metric': '{:.3f}',
    'p75_metric': '{:.3f}',
    'p90_metric': '{:.3f}',
    'top_n_mean_metric': '{:.3f}',
    'high_metric_share': '{:.1%}',
    'high_metric_threshold': '{:.3f}',
    'goal_share': '{:.1%}',
    'turnover_share': '{:.1%}',
}

TOP_POSSESSION_COLUMNS = [
    'rank', 'team_id', 'top_mean_metric', 'top_floor_metric', 'top_avg_throw_count',
    'team_aec_per_throw', 'median_possession_aec_per_throw', 'possessions', 'throws',
]

CONSISTENCY_COLUMNS = [
    'rank', 'team_id', 'median_metric', 'p75_metric', 'top_n_mean_metric',
    'high_metric_share', 'team_aec_per_throw', 'possessions', 'throws', 'avg_throw_count',
    'goal_share', 'turnover_share',
]


def show_team_aec_t_table(leaderboard, n=TOP_TEAMS, columns=LEADERBOARD_COLUMNS):
    table = leaderboard.head(n).reindex(columns=columns).copy()
    for column, formatter in FORMATTERS.items():
        if column in table:
            values = pd.to_numeric(table[column], errors='coerce')
            table[column] = values.map(lambda value: '' if pd.isna(value) else formatter.format(value))
    return table


POSITIVE_COLOR_SCALE = [
    [0.0, '#eaf4e8'],
    [0.35, '#9fd391'],
    [0.70, '#3f9c5a'],
    [1.0, '#09552f'],
]
DIVERGING_COLOR_SCALE = [
    [0.0, '#ba4a3c'],
    [0.50, '#f4f7f2'],
    [1.0, '#0b5d3b'],
]
HOVER_FORMATS = {
    'team_aec_per_throw': ':.3f',
    'avg_possession_aec_per_throw': ':.3f',
    'median_possession_aec_per_throw': ':.3f',
    'top_mean_metric': ':.3f',
    'top_floor_metric': ':.3f',
    'median_metric': ':.3f',
    'p75_metric': ':.3f',
    'top_n_mean_metric': ':.3f',
    'high_metric_share': ':.1%',
    'possessions': ':,',
    'games': ':,',
    'throws': ':,',
    'huck_rate': ':.1%',
    'o_line_share': ':.1%',
    'goal_share': ':.1%',
    'turnover_share': ':.1%',
}
HOVER_LABELS = {
    'team_aec_per_throw': 'team aEC/T',
    'avg_possession_aec_per_throw': 'avg possession aEC/T',
    'median_possession_aec_per_throw': 'median possession aEC/T',
    'top_mean_metric': 'top mean',
    'top_floor_metric': 'top floor',
    'median_metric': 'median',
    'p75_metric': '75th percentile',
    'top_n_mean_metric': 'top-N mean',
    'high_metric_share': 'high-rate share',
    'possessions': 'possessions',
    'games': 'games',
    'throws': 'throws',
    'huck_rate': 'huck rate',
    'o_line_share': 'O-line share',
    'goal_share': 'goal share',
    'turnover_share': 'turnover share',
}
TEAM_LOGOS = {
    'alleycats': 'https://watchufa.com/sites/default/files/IND-Logo-Teams.png',
    'apex': 'https://watchufa.com/sites/default/files/Apex-HeaderLogo.png',
    'bighorns': 'https://watchufa.com/sites/default/files/logo-team-bighorns.png',
    'breeze': 'https://watchufa.com/sites/default/files/logo-team-DC.png',
    'cascades': 'https://watchufa.com/sites/default/files/logo-team-SEA.png',
    'empire': 'https://watchufa.com/sites/default/files/logo-team-NY.png',
    'flyers': 'https://watchufa.com/sites/default/files/logo-team-RAL_0.png',
    'glory': 'https://watchufa.com/sites/default/files/medium_logo-team-BOS.png',
    'growlers': 'https://watchufa.com/sites/default/files/logo-team-SD_0.png',
    'havoc': 'https://watchufa.com/sites/default/files/medium_logo-team_HTX.png',
    'hustle': 'https://watchufa.com/sites/default/files/medium_logo-team-ATL.png',
    'phoenix': 'https://watchufa.com/sites/default/files/logo-team-PHI.png',
    'radicals': 'https://watchufa.com/sites/default/files/logo-team-MAD.png',
    'royal': 'https://watchufa.com/sites/default/files/logo-team-MTL.png',
    'rush': 'https://watchufa.com/sites/default/files/logo-team-TOR_0.png',
    'shred': 'https://watchufa.com/sites/default/files/ShredWeb.png',
    'sol': 'https://watchufa.com/sites/default/files/logo-team-AUS_0.png',
    'spiders': 'https://watchufa.com/themes/AUDL_theme/css/images/logos/logo-team-SJ.png',
    'steel': 'https://watchufa.com/sites/default/files/Org_team_page.png',
    'thunderbirds': 'https://watchufa.com/sites/default/files/logo-team-PIT.png',
    'union': 'https://watchufa.com/sites/default/files/logo-team-CHI_0.png',
    'windchill': 'https://watchufa.com/sites/default/files/logo-team-MIN.png',
}
LOGO_SOURCE_CACHE = {}
TEAM_LOGO_CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'team_logos'
LOGO_MIME_TYPES = {
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.png': 'image/png',
    '.svg': 'image/svg+xml',
    '.webp': 'image/webp',
}


def get_team_logo_source(team_id):
    team_id = str(team_id).lower()
    if team_id in LOGO_SOURCE_CACHE:
        return LOGO_SOURCE_CACHE[team_id]
    logo_url = TEAM_LOGOS.get(team_id)
    if not logo_url:
        return None
    suffix = Path(logo_url.split('?')[0]).suffix.lower() or '.png'
    cache_path = TEAM_LOGO_CACHE_DIR / f'{team_id}{suffix}'
    try:
        if not cache_path.exists():
            TEAM_LOGO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
            response = requests.get(logo_url, timeout=10)
            response.raise_for_status()
            cache_path.write_bytes(response.content)
        mime_type = LOGO_MIME_TYPES.get(cache_path.suffix.lower()) or mimetypes.guess_type(cache_path.name)[0] or 'image/png'
        logo_source = 'data:' + mime_type + ';base64,' + base64.b64encode(cache_path.read_bytes()).decode('ascii')
    except Exception:
        logo_source = logo_url
    LOGO_SOURCE_CACHE[team_id] = logo_source
    return logo_source


def plot_team_metric_leaderboard(leaderboard, metric, title, x_label, n=TOP_TEAMS):
    chart = leaderboard.head(n).sort_values(metric, ascending=True).copy()
    if chart.empty:
        return px.bar(title=title)
    chart['team_label'] = chart['rank'].astype(int).astype(str) + '. ' + chart['team_id'].astype(str).str.title()
    hover_columns = [
        column for column in [
            'team_aec_per_throw', 'avg_possession_aec_per_throw', 'median_possession_aec_per_throw',
            'top_mean_metric', 'top_floor_metric', 'median_metric', 'p75_metric',
            'top_n_mean_metric', 'high_metric_share', 'possessions', 'games', 'throws',
            'huck_rate', 'o_line_share', 'goal_share', 'turnover_share',
        ]
        if column in chart and column != metric
    ]
    metric_min = pd.to_numeric(chart[metric], errors='coerce').min()
    metric_max = pd.to_numeric(chart[metric], errors='coerce').max()
    has_negative_values = metric_min < 0
    crosses_zero = metric_min < 0 < metric_max
    color_scale = DIVERGING_COLOR_SCALE if has_negative_values else POSITIVE_COLOR_SCALE
    padding = max((metric_max - metric_min) * 0.10, 0.015)
    data_span = max(metric_max - min(metric_min, 0), 0.10)
    logo_space = max(data_span * 0.055, 0.010)
    x_range = [metric_min - padding - logo_space, metric_max + padding] if has_negative_values else [-logo_space, metric_max + padding]
    logo_x = x_range[0] + logo_space * 0.58
    logo_size_x = logo_space * 0.70
    hover_template = f'{x_label}=%{{x:.3f}}'
    for index, column in enumerate(hover_columns):
        label = HOVER_LABELS.get(column, column.replace('_', ' '))
        hover_template += f'<br>{label}=%{{customdata[{index}]{HOVER_FORMATS.get(column, "")}}}'
    hover_template += '<extra></extra>'
    fig = px.bar(
        chart,
        x=metric,
        y='team_label',
        orientation='h',
        text=metric,
        custom_data=hover_columns,
        labels={metric: x_label, 'team_label': 'Team'},
        color=metric,
        color_continuous_scale=color_scale,
    )
    fig.update_traces(
        texttemplate='%{x:.3f}',
        textposition='outside',
        cliponaxis=False,
        marker_line_color='rgba(255,255,255,0.85)',
        marker_line_width=0.8,
        opacity=0.96,
        hovertemplate=hover_template,
    )
    if crosses_zero:
        fig.add_vline(x=0, line_width=1, line_dash='dot', line_color='#6b7280')
        fig.update_coloraxes(cmid=0)
    for _, row in chart.iterrows():
        logo_source = get_team_logo_source(row['team_id'])
        if logo_source:
            fig.add_layout_image(
                dict(
                    source=logo_source,
                    xref='x',
                    yref='y',
                    x=logo_x,
                    y=row['team_label'],
                    sizex=logo_size_x,
                    sizey=0.72,
                    xanchor='center',
                    yanchor='middle',
                    layer='above',
                )
            )
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        title_font={'size': 20, 'color': '#20385f'},
        height=max(560, 32 * len(chart) + 150),
        width=1040,
        margin={'l': 44, 'r': 78, 't': 78, 'b': 62},
        coloraxis_showscale=False,
        bargap=0.24,
        font={'family': 'Segoe UI, Arial, sans-serif', 'size': 12, 'color': '#23395d'},
        hoverlabel={
            'bgcolor': '#ffffff',
            'bordercolor': '#c9d8c4',
            'font': {'color': '#14213d', 'size': 12, 'family': 'Segoe UI, Arial, sans-serif'},
        },
        plot_bgcolor='#f7fbf5',
        paper_bgcolor='#ffffff',
    )
    fig.update_xaxes(
        title_text=x_label,
        range=x_range,
        showgrid=True,
        gridcolor='#e5eee2',
        zeroline=False,
        linecolor='#d7e4d2',
        tickfont={'color': '#31476b'},
        title_font={'color': '#31476b'},
    )
    fig.update_yaxes(
        title_text='',
        showgrid=False,
        showticklabels=True,
        tickmode='array',
        tickvals=chart['team_label'].tolist(),
        ticktext=(chart['rank'].astype(int).astype(str) + '.').tolist(),
        ticks='',
        tickfont={'color': '#20385f'},
    )
    return fig


def plot_team_aec_t_leaderboard(leaderboard, title, n=TOP_TEAMS):
    return plot_team_metric_leaderboard(
        leaderboard,
        metric='team_aec_per_throw',
        title=title,
        x_label='team aEC/T',
        n=n,
    )

## All Possessions Including Turnovers

This is the closest view to Shown Space team `Tot-aEC`: it includes scoring possessions, turnovers, and other built possessions. This is the view where Bighorns show up as negative overall.

In [ ]:
team_aec_t_all_possessions = summarize_team_aec_per_throw(
    league_all_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_possessions,
    title=f'{SEASON} UFA team aEC/T - all possessions including turnovers',
)

In [ ]:
show_team_aec_t_table(team_aec_t_all_possessions)

## Total aEC Sanity Check

Shown Space's `Tot-aEC` is a total, not an aEC/T rate. Sorting the all-possession summary by `total_aec` should line up with that table much more closely than the scoring-only views.

In [ ]:
team_total_aec_low_to_high = team_aec_t_all_possessions.sort_values('total_aec', ascending=True).reset_index(drop=True).copy()
team_total_aec_low_to_high['rank'] = range(1, len(team_total_aec_low_to_high) + 1)
show_team_aec_t_table(team_total_aec_low_to_high)

## All Scoring Possessions

This view only includes possessions that ended in goals. It answers a narrower question: when a team scores, how much aEC does that scoring possession create per throw? It should not be read as overall team aEC.

In [ ]:
team_aec_t_all_scoring = summarize_team_aec_per_throw(
    league_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_scoring,
    title=f'{SEASON} UFA team aEC/T - all scoring possessions',
)

In [ ]:
show_team_aec_t_table(team_aec_t_all_scoring)

## Top 5 Non-Huck Scoring Possessions

Raw top-five aEC/T can overrate one-throw or very short scores. This version excludes hucks and requires at least `TOP_POSSESSION_MIN_THROWS` throws before selecting each team's top five.

In [ ]:
non_huck_scoring_possessions = league_possessions[
    pd.to_numeric(league_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()

team_top5_non_huck_aec_t = summarize_team_top_possessions(
    non_huck_scoring_possessions,
    top_n=TOP_POSSESSION_COUNT,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=TOP_POSSESSION_MIN_THROWS,
)

plot_team_metric_leaderboard(
    team_top5_non_huck_aec_t,
    metric='top_mean_metric',
    title=f'{SEASON} UFA top {TOP_POSSESSION_COUNT} non-huck scoring possession aEC/T, min {TOP_POSSESSION_MIN_THROWS} throws',
    x_label=f'top {TOP_POSSESSION_COUNT} mean aEC/T',
)

In [ ]:
show_team_aec_t_table(team_top5_non_huck_aec_t, columns=TOP_POSSESSION_COLUMNS)

## Consistent Non-Huck Scoring aEC/T

This is still scoring-only, so turnovers are not included. It asks which teams repeatedly finish successful non-huck scores with high aEC/T. The next section adds turnovers back in.

In [ ]:
eligible_non_huck_scoring = non_huck_scoring_possessions[
    pd.to_numeric(non_huck_scoring_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_scoring['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_scoring_possessions,
    high_threshold=non_huck_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck scoring possessions: {len(eligible_non_huck_scoring):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck scoring aEC/T, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck scoring aEC/T',
)

In [ ]:
show_team_aec_t_table(team_non_huck_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

## Consistent Non-Huck aEC/T Including Turnovers

This uses all built possessions, filters out hucks, and includes turnovers. This is a better team-quality version of the consistency view because failed possessions count against the team.

In [ ]:
non_huck_all_possessions = league_all_possessions[
    pd.to_numeric(league_all_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()
eligible_non_huck_all = non_huck_all_possessions[
    pd.to_numeric(non_huck_all_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_all_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_all['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_all_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_all_possessions,
    high_threshold=non_huck_all_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck all possessions: {len(eligible_non_huck_all):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_all_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_all_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck aEC/T including turnovers, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck all-possession aEC/T',
)

In [ ]:
show_team_aec_t_table(team_non_huck_all_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

## Default Possession-Shape Pool

This uses the same default filter as the top-possession shape work: O-line scoring possessions, long-field starts, hucks excluded.

In [ ]:
analysis_possessions, analysis_paths = filter_analysis_possessions(
    league_possessions,
    league_paths,
)
team_aec_t_default_pool = summarize_team_aec_per_throw(
    analysis_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_default_pool,
    title=f'{SEASON} UFA team aEC/T - O-line non-huck long-field scores',
)

In [ ]:
show_team_aec_t_table(team_aec_t_default_pool)